# Chapitre 13 - Exercices 13.1 à 13.4

* **13.1** : K-means et EM pour un mélange gaussien à covariance $\sigma I$ ; limite $\sigma \to 0$.
* **13.2** : rayon médian du 1-voisinage (éq. (13.7)).
* **13.3** : le théorème de Cover et Hart (éq. (13.5) et (13.12)).
* **13.4** : transformations affines d'une image et vecteurs tangents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch13_prototypes as cp
set_style()
PURPLE = COLORS[6]

## Exercice 13.1 : K-means et EM pour un mélange à covariance scalaire

**Énoncé (reformulé).** On considère un mélange gaussien dont toutes les matrices de
covariance sont scalaires, $\Sigma_r = \sigma I$ pour $r = 1, \dots, R$, où $\sigma > 0$ est
**fixé** (connu). Discuter en détail l'analogie entre l'algorithme de regroupement K-means
et l'algorithme EM pour ajuster ce mélange. Montrer que lorsque $\sigma \to 0$, les deux
méthodes coïncident.

### Rappels

**Le modèle.** La densité est
$$f(x) = \sum_{r=1}^R \pi_r\, \phi_\sigma(x; \mu_r), \qquad
\phi_\sigma(x; \mu) = (2\pi\sigma)^{-p/2} \exp\Big(-\frac{\|x - \mu\|^2}{2\sigma}\Big),$$
avec des proportions $\pi_r > 0$, $\sum_r \pi_r = 1$. Ici $\sigma$ joue le rôle d'une
**variance** (c'est la notation du livre : $\Sigma_r = \sigma I$).

**K-means.** Minimise $J(C, \mu) = \sum_i \|x_i - \mu_{C(i)}\|^2$ en alternant
affectation au centre le plus proche et recalcul des centres comme moyennes.

**EM** (vu dans le cours 1, section 5.2) : on alterne le calcul des responsabilités
(étape E) et une estimation pondérée des paramètres (étape M) ; chaque itération fait
croître la log-vraisemblance $\ell(\mu, \pi) = \sum_i \log f(x_i)$.

### Les deux étapes d'EM pour ce modèle

**Étape E.** Par la formule de Bayes, la responsabilité de la composante $r$ pour
l'observation $i$ est la probabilité a posteriori que $x_i$ provienne de $r$ :
$$\gamma_{ir} = \frac{\pi_r\, \phi_\sigma(x_i; \mu_r)}{\sum_{s} \pi_s\, \phi_\sigma(x_i; \mu_s)}
= \frac{\pi_r \exp(-d_{ir}/2\sigma)}{\sum_s \pi_s \exp(-d_{is}/2\sigma)}, \qquad d_{ir} = \|x_i - \mu_r\|^2,$$
car le facteur $(2\pi\sigma)^{-p/2}$, commun à toutes les composantes, se simplifie.

**Étape M.** On maximise $\sum_i \sum_r \gamma_{ir}\big[\log \pi_r + \log \phi_\sigma(x_i; \mu_r)\big]$.
En $\mu_r$, seul compte $-\sum_i \gamma_{ir}\|x_i - \mu_r\|^2/(2\sigma)$, fonction concave
dont le gradient $\sum_i \gamma_{ir}(x_i - \mu_r)/\sigma$ s'annule en
$$\mu_r = \frac{\sum_i \gamma_{ir} x_i}{\sum_i \gamma_{ir}} .$$
En $\pi$ (sous la contrainte $\sum_r \pi_r = 1$, par un multiplicateur de Lagrange) :
$\pi_r = \frac1N \sum_i \gamma_{ir}$. Comme $\sigma$ est fixé, il n'est pas ré-estimé.